In [1]:
from langchain.agents import create_sql_agent
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.sql_database import SQLDatabase
from langchain.llms.openai import OpenAI
from langchain.agents import AgentExecutor

In [3]:
db_user = "student123"
db_password = "student321"
#db_host = "localhost:3306"
db_host = "rm-uf6z891lon6dxuqblqo.mysql.rds.aliyuncs.com:3306"
db_name = "action"
db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}")
db

In [10]:
from langchain.chat_models import ChatOpenAI
import os

# 从环境变量获取 dashscope 的 API Key
api_key = os.environ.get('DASHSCOPE_API_KEY')

# 通过LLM => 撰写SQL
llm = ChatOpenAI(
    temperature=0.01,
    #model="deepseek-v3",  
    model = "qwen-turbo-latest",
    openai_api_base = "https://dashscope.aliyuncs.com/compatible-mode/v1",
    openai_api_key  = api_key
)

In [11]:
# 需要设置llm
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# SQL智能体：给它目标，它自己会进行规划，最终把结果给你
agent_executor = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=True,
    handle_parsing_errors=True,  # 添加错误处理
    max_iterations=5,  # 限制最大迭代次数
    early_stopping_method="generate"  # 提前停止方法
)

In [6]:
agent_executor.run("描述与订单相关的表及其关系")

/tmp/ipykernel_28230/396545954.py:1: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  agent_executor.run("描述与订单相关的表及其关系")




> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: address, asset_grades, bank, car_sales, customers, dept, employee, form, height_grades, hero_score, heros, orders, person, player, player_score, student, team, team_score, test_work, trips, user, users, using, weatherI see several tables that might be related to orders, particularly "orders" and "customers". I should examine their schemas to understand their structure and relationships.

Action: sql_db_schema
Action Input: orders, customers
CREATE TABLE customers (
	`Id` INTEGER, 
	`Name` VARCHAR(255)
)DEFAULT CHARSET=utf8mb3 ENGINE=InnoDB

/*
3 rows from customers table:
Id	Name
1	Joe
2	Henry
3	Sam
*/


CREATE TABLE orders (
	`Id` INTEGER, 
	`CustomerId` INTEGER
)DEFAULT CHARSET=utf8mb3 ENGINE=InnoDB

/*
3 rows from orders table:
Id	CustomerId
1	3
2	1
*/From examining the schema, I can see that:

1. The `orders` table contains:
   - `Id` (order ID)
   - `CustomerId` (which references the customer who

'与订单相关的表主要有两个：\n1. `orders`表 - 存储订单信息，包含字段：Id(订单ID), CustomerId(客户ID)\n2. `customers`表 - 存储客户信息，包含字段：Id(客户ID), Name(客户姓名)\n\n它们之间的关系是：orders表中的CustomerId字段作为外键关联到customers表的Id字段，表示一个客户可以拥有多个订单的一对多关系。'

In [14]:
agent_executor.invoke({"input": "数据库中有哪些表"})



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: address, asset_grades, bank, car_sales, customers, dept, employee, form, height_grades, hero_score, heros, orders, person, player, player_score, student, team, team_score, test_work, trips, user, users, using, weatherI now know the tables available in the database. Since the question is asking about what tables exist, I can provide the list directly.

Final Answer: 数据库中有以下表：address, asset_grades, bank, car_sales, customers, dept, employee, form, height_grades, hero_score, heros, orders, person, player, player_score, student, team, team_score, test_work, trips, user, users, using, weather.

> Finished chain.


{'input': '数据库中有哪些表',
 'output': '数据库中有以下表：address, asset_grades, bank, car_sales, customers, dept, employee, form, height_grades, hero_score, heros, orders, person, player, player_score, student, team, team_score, test_work, trips, user, users, using, weather.'}

In [15]:
agent_executor.run("找出英雄攻击力最高的前5个英雄")



> Entering new SQL Agent Executor chain...
Action: sql_db_list_tables
Action Input: address, asset_grades, bank, car_sales, customers, dept, employee, form, height_grades, hero_score, heros, orders, person, player, player_score, student, team, team_score, test_work, trips, user, users, using, weatherLooking at the list of tables, "heros" seems to be the most relevant table for finding heroes and their attributes like attack power. I should check the schema of the "heros" table to understand its structure.
Action: sql_db_schema
Action Input: heros
CREATE TABLE heros (
	id INTEGER NOT NULL AUTO_INCREMENT, 
	name VARCHAR(255) CHARACTER SET utf8 COLLATE utf8_general_ci NOT NULL, 
	hp_max FLOAT, 
	hp_growth FLOAT, 
	hp_start FLOAT, 
	mp_max FLOAT, 
	mp_growth FLOAT, 
	mp_start FLOAT, 
	attack_max FLOAT, 
	attack_growth FLOAT, 
	attack_start FLOAT, 
	defense_max FLOAT, 
	defense_growth FLOAT, 
	defense_start FLOAT, 
	hp_5s_max FLOAT, 
	hp_5s_growth FLOAT, 
	hp_5s_start FLOAT, 
	mp_5s_max F

'阿轲, 孙尚香, 百里守约, 虞姬, 黄忠'